# 5장 — 축마다 다른 보폭: Adam ⚙️ (핵심)

Adam을 numpy 세 줄로 직접 조립하고, 지형이 100배 가파라져도 같은 설정으로 생존하는
**스케일 불변**을 재현합니다.

**이 노트북의 목표**
1. 두 지형 3파전: 지형 B(곡률 100배)에서 GD·모멘텀은 폭발, Adam만 생존
2. 스케일 지우개: RMS로 나누면 어떤 크기의 기울기도 ±1로 표준화
3. 편향 보정: 초반 EMA 과소평가와 m̂ = m/(1−β₁ᵗ)의 효과

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch5')

## 1. 실험 ①: 두 지형 3파전 (설정은 지형과 무관하게 고정)

타원 골짜기 L = ½·0.1·x² + ½·c_y·y², 출발 (−9, 1.5). GD·모멘텀 lr 0.2 / Adam lr 0.3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

CX = 0.1

def race(cy, steps=80):
    """세 옵티마이저로 같은 골짜기 경주 — 손실 이력 3개 반환 (결정적)."""
    L = lambda x, y: 0.5*CX*x*x + 0.5*cy*y*y
    hist = {'GD': [], '모멘텀': [], 'Adam': []}
    # GD
    x, y = -9.0, 1.5
    for _ in range(steps):
        x, y = x - 0.2*CX*x, y - 0.2*cy*y
        hist['GD'].append(min(L(x, y), 1e12))
    # 모멘텀
    x, y, mx, my = -9.0, 1.5, 0.0, 0.0
    for _ in range(steps):
        mx, my = 0.9*mx + CX*x, 0.9*my + cy*y
        x, y = x - 0.2*mx, y - 0.2*my
        hist['모멘텀'].append(min(L(x, y), 1e12))
    # Adam (세 줄의 조립품)
    x, y = -9.0, 1.5
    m = np.zeros(2); v = np.zeros(2)
    for t in range(1, steps + 1):
        g = np.array([CX*x, cy*y])
        m = 0.9*m + 0.1*g                       # 방향의 EMA (β₁ — 4장의 모멘텀)
        v = 0.999*v + 0.001*g*g                 # 크기의 EMA (β₂ — 오늘의 RMS)
        mh, vh = m/(1 - 0.9**t), v/(1 - 0.999**t)   # 편향 보정
        x, y = x - 0.3*mh[0]/(np.sqrt(vh[0]) + 1e-8), y - 0.3*mh[1]/(np.sqrt(vh[1]) + 1e-8)
        hist['Adam'].append(min(L(x, y), 1e12))
    return hist

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
for ax, cy in zip(axes, [8, 800]):
    for name, h in race(cy).items():
        ax.semilogy(np.maximum(h, 1e-8), label=name)
    ax.set_title(f'지형 벽 곡률 = {cy}'); ax.set_xlabel('스텝'); ax.legend(); ax.grid(alpha=0.3)
axes[0].set_ylabel('손실 (log)'); plt.tight_layout(); plt.show()

A, B = race(8), race(800)
print(f'지형 A 최종: GD {A["GD"][-1]:.4f} / 모멘텀 {A["모멘텀"][-1]:.4f} / Adam {A["Adam"][-1]:.4f}')
print(f'지형 B 최종: GD {B["GD"][-1]:.3g} / 모멘텀 {B["모멘텀"][-1]:.3g} / Adam {B["Adam"][-1]:.4f}')

assert A['Adam'][-1] < 0.01 and A['모멘텀'][-1] < 0.01, '지형 A: 모멘텀·Adam 모두 무난'
assert B['GD'][-1] > 1e6 and B['모멘텀'][-1] > 1e6, '지형 B: GD·모멘텀은 같은 설정 그대로 폭발 (경계 2/c ↓100배)'
assert B['Adam'][-1] < 0.5, '지형 B: Adam만 생존 — 스케일 불변'
print('\n✔ 확인: 지형이 100배 가파라져도 Adam은 같은 설정으로 그냥 간다. 기본값의 자격')

## 2. 실험 ② · 스케일 지우개: RMS로 나누면 크기가 사라진다

In [ ]:
def rms_normalize(A, n=60):
    """크기 A 근처의 기울기 흐름을 RMS(g²의 EMA의 √)로 나눈 마지막 값."""
    v, out = 0.0, 0.0
    for k in range(n):
        g = A * (1.0 if k % 2 == 0 else 0.8)     # 크기 A 근처에서 살짝 변하는 흐름
        v = 0.999*v + 0.001*g*g
        out = g / (np.sqrt(v / (1 - 0.999**(k+1))) + 1e-8)
    return out

print(' 원래 크기 A   나누기 후')
for A in [0.01, 0.5, 5, 500]:
    r = rms_normalize(A)
    print(f'  {A:8.2f}   {r:8.3f}')
    assert 0.7 < abs(r) < 1.3, '어떤 크기든 나누기 후엔 1 근처 — 크기가 지워진다'

print('\n✔ 확인: 5만 배 차이 나는 축들이 같은 자(±1)로. 이것이 축마다 다른 보폭 (7부 2장 정규화의 축별 판)')

## 3. 실험 ③ · 편향 보정: 첫 걸음부터 제 크기로

In [ ]:
g_true = 2.0                                     # 일정한 진짜 기울기
m = 0.0
print(' t    보정 전 m    보정 후 m̂   (진짜 기울기 = 2.0)')
for t in range(1, 6):
    m = 0.9*m + 0.1*g_true
    mh = m / (1 - 0.9**t)
    print(f' {t}    {m:8.3f}    {mh:8.3f}')
    assert abs(mh - g_true) < 1e-9, '보정 후에는 첫 걸음부터 정확히 제 크기'

m1 = 0.1 * g_true
assert abs(m1 - 0.2) < 1e-12, '보정 전 t=1의 m은 진짜의 10분의 1 — EMA가 0에서 출발한 탓'
print('\n✔ 확인: m̂ = m/(1−β₁ᵗ). 초반 과소평가를 되살리는 자동 예열 장치')

## 4. 관찰 정리 + 🤖 AI 연결

- **Adam = 모멘텀(β₁, 방향의 EMA) + RMS(β₂, 크기의 EMA)**: 방향을 평소 크기로 나눠 걷는다.
  유효 보폭 = lr ÷ 평소 크기 → 축마다 자동 맞춤(2장 딜레마 해결).
- **스케일 불변**: 지형이 100배 가파라져도 같은 설정으로 생존. lr 튜닝이 관대해지는 것이
  Adam이 기본값인 이유. (단 초반 v̂ 추정이 서기 전은 취약 → warmup, 6장.)
- 설정 정체: β₁=0.9(기억 10걸음), β₂=0.999(1000걸음, 자는 천천히), ε=1e-8(0 나누기 안전핀),
  편향 보정 = 자동 예열. 청구서: 파라미터당 m·v 저장, 옵티마이저 상태가 모델 무게의 2배.
- 다시 쓴 것: 4장 EMA 두 개 + 1부 1장 좌표축(축마다 다른 자). 실무 이름은 대부분 AdamW(8장).

**다음 챕터**: 6장(여행 일정표). 보폭을 여행 단계마다 바꾸는 스케줄(워밍업·코사인 감쇠),
설정 파일 `warmup_steps`·`lr_scheduler`의 정체.